# Collecte automatisée en python des cours monétaires en 2024-2025

- dates de révision :

## Binôme : 
- nom1 prenom1 adresse UBS couriel1 (implication dans collecte : - [X] Oui - [ ] Non) ;
- nom2 prenom2 adresse UBS couriel2 (implication dans collecte : - [ ] Oui - [ ] Non).

## Expérience préalable dans la collecte automatisée d'une API ou de web services
Si impliqué dans la collecte, (reproduire le paragraphe si aussi deuxième binôme)
- nom1 prenom1 : [ ] j'ai refait voire reproduit et étudier attentivement les exemples du cours concernant :
    - [ ] `urllib` - [ ] `requests` - [ ] `re` - [ ] `lxml` - [ ] `bs4` - [ ] scrapy - [ ] `selenium` - [ ] scrapy-selenium 
- J'ai étudié exhaustivement la documentation officielle de :
    - [ ] `urllib` - [ ] `requests` - [ ] `re` - [ ] `lxml` - [ ] `bs4` - [ ] scrapy - [ ] `selenium` - [ ] scrapy-selenium 
- J'ai déjà collecté des informations sur X sites utilisant simplement html comme moteur de rendu de l'information (voir les fiches Y, Z)
- J'ai déjà collecté des informations sur X sites utilisant javascript / AJAX comme moteur de rendu de l'information (voir les fiches Y, Z)
- J'ai déjà été confronté à des mesures de rétorsion / défense de sites comme :
    - [ ] banissement IP - [ ] Captcha  - [ ] Obligation de s'identifier via un compte - [ ] Autre : préciser

## Site de cours monétaires :
- URL :
- Description du site :
- URL de la page d'aide du site pour la collecte de cours :
- Date de l'étude de ce site : 
- Technologie de rendu des informations clés à récupérer suite à l'analyse préalable :
   - [ ] HTML - [ ] Javascript - [x] API REST - [ ] Web Service SOAP

Si le site ne donne pas directement le cours de la monnaie fluctuante par rapport à l'euro, donnez la formule de calcul de ce cours fuctuant pour les deux monnaies ques vous avez retenu.

## Analyse préalable :
- Qualification du cas : - [ ] Simple car API REST dédiée - [ ] Simple car WS SOAP dédié [ ] Complexe 
- Outils utilisés : - [ ] Inspecter de FireFox - [x] Inspecter de Chrome - [ ] scrapy shell
- Sélecteurs utilisés donc envisagés pour la programmation : - [ ] CSS - [ ] Xpath
- Modules python pour la collecte de document : - [ ] `urllib` - [ ] `requests` - [ ] `selenium` - [ ] j'utilise srapy 
- Modules python pour l'analyse de document : - [ ] `re` - [ ] `lxml` - [ ] `bs4` - [ ] `selenium` - [ ] j'utilise srapy

- Pour chacune des informations recherchées, décrire en français votre stratégie de collecte puis exprimez cette stratégie sous forme opérationnelle (règle, expression bs4, expression réguliere, utilisation de fonction dans la scrapy shell, etc.

Suite à la récupération de ces cours monétaires journaliers, il faudrait les stocker dans une base de données MySQL. Une première étape consiste à sauvegarder les données collectées dans un fichier CSV qui est conforme au schema suivant :
```
DROP TABLE IF EXISTS t_cours;
DROP TABLE IF EXISTS t_devise;

CREATE TABLE IF NOT EXISTS t_devise(
id_devise INT AUTO_INCREMENT PRIMARY KEY  COMMENT 'Identifiant de la devise',
lib_devise VARCHAR(10) COMMENT 'Libelle de la devise',
isocode VARCHAR(3) UNIQUE COMMENT 'Code ISO de la devise',
symbole VARCHAR(3) COMMENT 'Symbole de la devise',
format_bo VARCHAR(20) COMMENT 'format dans la syntaxe SAP BusinessObjects de la devise',
cours_fixe NUMERIC COMMENT 'Cours fixe de la devise par rapport a l''euro',
CONSTRAINT CHK_Currency_Code_1 CHECK ( isocode REGEXP '[A-Z][A-Z][A-Z]' ),
CONSTRAINT CHK_Currency_Code_2 CHECK ( isocode = UPPER( isocode ) )
) ENGINE=InnoDB DEFAULT CHARSET=UTF8MB4 COMMENT 'Les devises d''interet pour le groupe DARTIES';

INSERT INTO t_devise VALUES(1,'Euro','EUR','€',NULL,1);
INSERT INTO t_devise VALUES(2,'Dollar','USD','$',NULL,NULL);
INSERT INTO t_devise VALUES(3,'Yen','JPY','¥',NULL,NULL);
INSERT INTO t_devise VALUES(4,'Franc','FRF','F',NULL,6.55957);

CREATE TABLE IF NOT EXISTS t_cours(
id_devise INT NOT NULL  COMMENT 'Cle etrangere vers T_DEVISE',
pubdate DATE NOT NULL  COMMENT 'Date de publication du cours fluctuant',
jour NUMERIC COMMENT 'Jour de la date de publication du cours fluctuant',
mois NUMERIC COMMENT 'Mois de la date de publication du cours fluctuant',
annee NUMERIC COMMENT 'Annee de la date de publication du cours fluctuant',
cours NUMERIC(28,20) COMMENT 'cours de la devise a la cloture de la date de publication',
CONSTRAINT t_cours_pkey PRIMARY KEY(id_devise,pubdate),
CONSTRAINT t_cours_uq UNIQUE(id_devise,jour,mois,annee),
INDEX t_cours_i_fkey_idx (id_devise),
CONSTRAINT t_cours_i_fkey FOREIGN KEY(id_devise) REFERENCES t_devise(id_devise)
) ENGINE=InnoDB DEFAULT CHARSET=UTF8MB4 COMMENT 'Les cours fluctuants des devises d''interet pour le groupe DARTIES';
```

ici, ce sont les Yen et Dollar qui sont des monnaies fluctuantes par rapport à l'Euro.

NB : On dispose d'un historique journalier des cours pour ces deux monnaies du 01 janvier 2019 au 31 décembre 2022.

Si vous choisissez une autre monnaie comme le Yuan par exemple, il vous faudra les valeurs par rapport à l'euro du 01 janvier 2019 à la fin de la période.
La fin de la période pour la SAE 301 2024-2025 est le 30 novembre 2024.

## Programmation de la collecte :
Pour toutes les technologies utilisées, on a vu qu'elles pouvaient être lancée depuis un notebook jupyter. Par contre pour une programmation pro scrapy exige un projet bien structuré. Il s'agit là de donner un programme qui ne récupère que les dix premiers exemplaires des informations recherchées. Mettre un lien vers l'archive contenant les programmes qui font la collecte exhaustive.

La collecte s'est elle déroulée en plusieurs fois ? - [ ] non - [ ] oui

Si non, date de la collecte exhaustive et monnaies, début de la période, fin de la période pour les données collectées :

Si oui, dates et pour chacune d'elles monnaies, début de la période, fin de la périodes :


### Programme qui récupère les dix premiers cours

In [ ]:
import requests
import csv

annees = list(range(2019, 2026))

with open("usd_jpy.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["DATE", "CoursJPY"])  

    for annee in annees:
        start_date = str(annee) + "-01-01"
        end_date = str(annee) + "-12-31"
        url = "https://api.exchangerate.host/timeframe"
        params = {
            "start_date": start_date,
            "end_date": end_date,
            "base": "USD",
            "symbols": "JPY",
            "access_key": "VOTRE_CLE_API"
        }

        response = requests.get(url, params=params)
        if response.status_code == 200:
            data = response.json()
            if "quotes" in data:
                for jour in sorted(data["quotes"].keys()):
                    taux = data["quotes"][jour]["USDJPY"]
                    writer.writerow([jour, taux])
            else:
                print("Pas de 'quotes' pour l'année", annee)
        else:
            print("Erreur autre pour l'année", annee)

print("CSV généré : usd_jpy.csv")


## Formule 
Dans ce calepin, vous proposerez comment utiliser des cours moyens mensuels pour convertir les rapports dans une des deux monnaies locales hors Euro. On attend de votre par une formule par rapport à la grandeur monétaire (CA, marge, coûts, salaire) en euro stockées dans la base de données.. Une requête SQL SELECT ... FROM renvoyant un historique multi-mensuel sera donnée à titre d'exemple pour expliciter votre formule.

### Test de connexion au SGBDR
Démarrez MySQL 8.0.18 via le raccourci `mysql-8.0.18-winx64 mysql_start.bat - Lecteur D` dans `D:\tools`.

`show tables` est ici une requête SQL qui est exécuté via le client lourd console `mysql`. Le port 3312 est le port du serveur local MySQL 8.0.18 à ne pas confondre avec le serveur MariaDB de la distribution Xampp (port=3314). 


In [ ]:
!D:\tools\xampp-7.2.15-0-VC15\mysql\bin\mysql -P 3312 -u python -ppython e_23_3jexxxxxxx -e "show tables"

### Un rapport d'un historique multi-mensuel d'une filiale américaine monétaire

Une requête SQL sur une grandeur monétaire (Chiffre d'affaires, marge, coûts, salaire).
On va calculer d'abord les cours moyens mensuels euro - dollar (1€ donne ? $) 

In [ ]:
!D:\tools\xampp-7.2.15-0-VC15\mysql\bin\mysql -P 3312 -u python -ppython e_23_3jexxxxxxx -e "SELECT annee, mois, AVG(cours) as cours_moyens_mensuels_euro_dollar FROM t_cours WHERE id_devise=2 GROUP BY annee, mois ORDER BY 1,2"

Actuellement le groupe DARTIES possède 48 magasins tous en France.

Soit le magasin hypothétique n°49 qui se trouve aux états-unis avec un client américain n°1611 qui n'a passé que deux commandes (à des mois différents).
On va utiliser les [*Common Table Expression*](https://dev.mysql.com/doc/refman/8.4/en/with.html) ou CTE `USA_FACTURE`, `USA_LIGNE_FACTURE` et `USA_COURS_MENSUELS` qui créent des jeux de données uniquement visibles dans la requête.

Voici l'ébauche de la requête SQL du rapport :
```
WITH USA_FACTURE AS (
SELECT 'CDE-201901010100731' num_commande,	'EUR' code_devise, 1611 id_client, 35 id_employe, 49 id_magasin, '2019-01-01' date_commande_client, '2019-01-31' date_livraison_client, 	'Due on order' condition_reglement,	'Credit card' mode_reglement, 	'Commande validée' statut_commande, 'Colissimo Suivi' mode_livraison, 	'3 weeks' disponibilite,  	'Mailing campaign' channel UNION ALL
SELECT 'CDE-201901010111111' num_commande,	'EUR' code_devise, 1611 id_client, 35 id_employe, 49 id_magasin, '2019-02-01' date_commande_client, '2019-02-28' date_livraison_client, 	'Due on order' condition_reglement,	'Credit card' mode_reglement, 	'Commande validée' statut_commande, 'Colissimo Suivi' mode_livraison, 	'3 weeks' disponibilite,  	'Mailing campaign' channel
),
USA_LIGNE_FACTURE AS (
SELECT 'CDE-201901010100731' num_commande, 1 id_ligne, 0.2000000000 taux_tva, 1.00 quantite, 219.98 pu_ht, 175.98 prix_revient, 0.00 reduction, 'EUR' code_devise, 11 id_produit UNION ALL
SELECT 'CDE-201901010111111' num_commande, 1 id_ligne, 0.2000000000 taux_tva, 5.00 quantite, 398.97 pu_ht, 319.18 prix_revient, 0.00 reduction, 'EUR' code_devise, 233 id_produit UNION ALL
SELECT 'CDE-201901010111111' num_commande, 2 id_ligne, 0.2000000000 taux_tva, 2.00 quantite, 219.98 pu_ht, 175.98 prix_revient, 0.00 reduction, 'EUR' code_devise, 11 id_produit
), 
USA_COURS_MENSUELS AS (
SELECT annee, mois, AVG(cours) as cours_moyens_mensuels_euro_dollar FROM t_cours WHERE id_devise=2 GROUP BY annee, mois ORDER BY 1,2
)
SELECT id_client, id_magasin, USA_FACTURE.num_commande, USA_FACTURE.date_commande_client, USA_COURS_MENSUELS.cours_moyens_mensuels_euro_dollar, id_ligne, t_produit.libelle, quantite, pu_ht as prix_euro, pu_ht as prix_dollar, prix_revient as cout_euro, prix_revient as cout_dollar, (pu_ht*quantite)*(1-reduction) as total_euros, (pu_ht*quantite)*(1-reduction) as total_dollar   
FROM USA_FACTURE
INNER JOIN USA_LIGNE_FACTURE
ON USA_LIGNE_FACTURE.num_commande=USA_FACTURE.num_commande
INNER JOIN t_produit
ON USA_LIGNE_FACTURE.id_produit=t_produit.id_produit
CROSS JOIN USA_COURS_MENSUELS
WHERE YEAR(date_commande_client)=USA_COURS_MENSUELS.annee
AND MONTH(date_commande_client)=USA_COURS_MENSUELS.mois
;
```
Bien entendu, il faut amender cette ébauche pour utiliser `cours_moyens_mensuels_euro_dollar` au niveau des trois grandeurs monétaires en dollar. A vous de le faire !

Completez la commande ci-dessous avec votre requête SQL :

In [ ]:
!D:\tools\xampp-7.2.15-0-VC15\mysql\bin\mysql -P 3312 -u python -ppython e_23_3jexxxxxxx -e ""

A noter que l'exécution de l'ébauche de la requête donne :

In [ ]:
!D:\tools\xampp-7.2.15-0-VC15\mysql\bin\mysql -P 3312 -u python -ppython e_23_3jexxxxxxx -e "WITH USA_FACTURE AS (SELECT 'CDE-201901010100731' num_commande,	'EUR' code_devise, 1611 id_client, 35 id_employe, 49 id_magasin, '2019-01-01' date_commande_client, '2019-01-31' date_livraison_client, 	'Due on order' condition_reglement,	'Credit card' mode_reglement, 	'Commande validée' statut_commande, 'Colissimo Suivi' mode_livraison, 	'3 weeks' disponibilite,  	'Mailing campaign' channel UNION ALL SELECT 'CDE-201901010111111' num_commande,	'EUR' code_devise, 1611 id_client, 35 id_employe, 49 id_magasin, '2019-02-01' date_commande_client, '2019-02-28' date_livraison_client, 	'Due on order' condition_reglement,	'Credit card' mode_reglement, 	'Commande validée' statut_commande, 'Colissimo Suivi' mode_livraison, 	'3 weeks' disponibilite,  	'Mailing campaign' channel), USA_LIGNE_FACTURE AS (SELECT 'CDE-201901010100731' num_commande, 1 id_ligne, 0.2000000000 taux_tva, 1.00 quantite, 219.98 pu_ht, 175.98 prix_revient, 0.00 reduction, 'EUR' code_devise, 11 id_produit UNION ALL SELECT 'CDE-201901010111111' num_commande, 1 id_ligne, 0.2000000000 taux_tva, 5.00 quantite, 398.97 pu_ht, 319.18 prix_revient, 0.00 reduction, 'EUR' code_devise, 233 id_produit UNION ALL SELECT 'CDE-201901010111111' num_commande, 2 id_ligne, 0.2000000000 taux_tva, 2.00 quantite, 219.98 pu_ht, 175.98 prix_revient, 0.00 reduction, 'EUR' code_devise, 11 id_produit), USA_COURS_MENSUELS AS (SELECT annee, mois, AVG(cours) as cours_moyens_mensuels_euro_dollar FROM t_cours WHERE id_devise=2 GROUP BY annee, mois ORDER BY 1,2) SELECT id_client, id_magasin, USA_FACTURE.num_commande, USA_FACTURE.date_commande_client, USA_COURS_MENSUELS.cours_moyens_mensuels_euro_dollar, id_ligne, t_produit.libelle, quantite, pu_ht as prix_euro, pu_ht as prix_dollar, prix_revient as cout_euro, prix_revient as cout_dollar, (pu_ht*quantite)*(1-reduction) as total_euros, (pu_ht*quantite)*(1-reduction) as total_dollar FROM USA_FACTURE INNER JOIN USA_LIGNE_FACTURE ON USA_LIGNE_FACTURE.num_commande=USA_FACTURE.num_commande INNER JOIN t_produit ON USA_LIGNE_FACTURE.id_produit=t_produit.id_produit CROSS JOIN USA_COURS_MENSUELS WHERE YEAR(date_commande_client)=USA_COURS_MENSUELS.annee AND MONTH(date_commande_client)=USA_COURS_MENSUELS.mois;"

Cette fiche doit être complétée et présente dans l'archive de rendu au niveau du répertoire `5-Monnaie`